## 数据集简要说明

ETT = Electricity Transformer Temperature，电力变压器温度数据集，时间范围：2016-07 ~ 2018-07，一共 7 个变量：
`date, HUFL, HULL, MUFL, MULL, LUFL, LULL, OT`

- OT：油温（预测目标）
- HUFL/HULL/MUFL/MULL/LUFL/LULL：高低压侧负载特征

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# 兼容从项目根目录或 Lesson4 目录启动 notebook
project_root = Path.cwd()
if not (project_root / "train_cuda.py").is_file():
    project_root = project_root.parent
if not (project_root / "train_cuda.py").is_file():
    raise FileNotFoundError("请从项目根目录或 Lesson4 目录启动 notebook")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from train_cuda import train

In [3]:
# ---------------------- 数据集类：滑动窗口构造时序样本 ----------------------
class ETTDataset(Dataset):
    def __init__(self, data, lookback=96, pred_len=24):
        """
        data: 归一化后的数组 [T, 7]
        lookback: 用前面多少步历史
        pred_len: 预测未来多少步 OT
        """
        self.lookback = lookback
        self.pred_len = pred_len
        self.data = data
        self.samples = []
        self.labels = []

        T = len(data)
        for i in range(T - lookback - pred_len + 1):
            x_seq = data[i:i+lookback, :]       # [lookback, 7]
            y_seq = data[i+lookback:i+lookback+pred_len, -1:]  # 只预测最后一列 OT
            self.samples.append(x_seq)
            self.labels.append(y_seq.squeeze(-1))

        self.samples = np.array(self.samples, dtype=np.float32)
        self.labels = np.array(self.labels, dtype=np.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.samples[idx])
        y = torch.from_numpy(self.labels[idx])
        return x, y

In [4]:
# ---------------------- 数据加载、归一化、划分 ----------------------
def load_ett(csv_path="ETTh1.csv", lookback=96, pred_len=24):
    df = pd.read_csv(csv_path)
    # 去掉date，只保留7个数值特征
    raw_data = df[["HUFL","HULL","MUFL","MULL","LUFL","LULL","OT"]].values

    # 时序划分：训练/验证/测试（标准划分）
    train_size = int(0.7 * len(raw_data))
    val_size = int(0.1 * len(raw_data))
    test_size = len(raw_data) - train_size - val_size

    train_data = raw_data[:train_size]
    val_data   = raw_data[train_size:train_size+val_size]
    test_data  = raw_data[train_size+val_size:]

    # 用训练集统计量做归一化（重要！不能泄露测试集信息）
    mean = train_data.mean(axis=0)
    std  = train_data.std(axis=0)
    train_data = (train_data - mean) / std
    val_data   = (val_data - mean) / std
    test_data  = (test_data - mean) / std

    train_ds = ETTDataset(train_data, lookback, pred_len)
    val_ds   = ETTDataset(val_data, lookback, pred_len)
    test_ds  = ETTDataset(test_data, lookback, pred_len)

    return train_ds, val_ds, test_ds, mean, std

In [6]:
lookback = 96
pred_len = 24
batch_size = 32
lr = 1e-3
epochs = 15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

csv_path = project_root / "Datasets" / "ETTh1.csv"
train_ds, val_ds, test_ds, mean, std = load_ett(csv_path, lookback, pred_len)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

## 训练多步预测模型

GRU 使用过去 96 小时的 7 个特征，一次输出未来 24 小时的油温；预测时不需要未来的负载值。损失使用训练集标准化后的油温，最后再还原到原始温度单位。

In [ ]:
class OilTemperatureGRU(nn.Module):
    def __init__(self, input_size=7, hidden_size=64, pred_len=24):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, pred_len)

    def forward(self, x):
        sequence, _ = self.gru(x)
        return self.output(sequence[:, -1, :])

torch.manual_seed(23)
model = OilTemperatureGRU(pred_len=pred_len).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
print(f"Training on {device}")
losses, val_losses = train(model, train_loader, val_loader, loss_fn, optimizer, epochs, device)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses, label="Train")
plt.plot(val_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("MSE (normalized)")
plt.legend()
plt.grid(True)
plt.show()

## 测试集评估及逐小时预测

测试集上的 MAE 和 RMSE 汇总所有滑动窗口及预测步长，因此同一时刻可能计入多次。下图展示测试集第一个窗口的 24 小时预测与真实值；它们不参与训练。

In [ ]:
model.eval()
absolute_error = 0.0
squared_error = 0.0
count = 0
with torch.inference_mode():
    for x_batch, y_batch in test_loader:
        prediction = model(x_batch.to(device)).cpu()
        error = (prediction - y_batch) * std[-1]
        absolute_error += error.abs().sum().item()
        squared_error += error.square().sum().item()
        count += error.numel()
print(f"Test MAE: {absolute_error / count:.3f}, RMSE: {np.sqrt(squared_error / count):.3f}")

x_test, y_test = test_ds[0]
with torch.inference_mode():
    y_pred = model(x_test.unsqueeze(0).to(device)).cpu().numpy()[0]
y_pred = y_pred * std[-1] + mean[-1]
y_true = y_test.numpy() * std[-1] + mean[-1]
dates = pd.to_datetime(pd.read_csv(csv_path, usecols=["date"])["date"])
test_start = int(0.7 * len(dates)) + int(0.1 * len(dates))
test_dates = dates.iloc[test_start + lookback:test_start + lookback + pred_len].reset_index(drop=True)
test_forecast = pd.DataFrame({"date": test_dates, "actual_OT": y_true, "predicted_OT": y_pred})
display(test_forecast)

plt.figure(figsize=(10, 4))
plt.plot(test_dates, y_true, label="Actual")
plt.plot(test_dates, y_pred, label="Predicted")
plt.xlabel("Time")
plt.ylabel("Oil temperature")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 预测数据集之后的油温

使用数据集中最后 96 小时的观测值，逐小时列出接下来 24 小时的预测。这里是一次性多步预测，不是把预测油温递归地作为下一步输入；未来真实值未知，预测误差不能在此直接计算。

In [ ]:
features = ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT"]
df = pd.read_csv(csv_path, parse_dates=["date"])
latest = df[features].tail(lookback).to_numpy()
latest_normalized = (latest - mean) / std
latest_tensor = torch.tensor(latest_normalized, dtype=torch.float32, device=device).unsqueeze(0)
with torch.inference_mode():
    future_ot = model(latest_tensor).cpu().numpy()[0] * std[-1] + mean[-1]
future_dates = pd.date_range(df["date"].iloc[-1] + pd.Timedelta(hours=1), periods=pred_len, freq="h")
future_forecast = pd.DataFrame({"date": future_dates, "predicted_OT": future_ot})
display(future_forecast)

plt.figure(figsize=(10, 4))
plt.plot(future_dates, future_ot, marker="o", label="Forecast")
plt.xlabel("Time")
plt.ylabel("Oil temperature")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()